From the `Bamboo_setup` directory, run the following command:
```bash
jupyter notebook --no-browser --port=8888
```
Copy the URL starting with `http://localhost:8888/`.
When picking the kernel for this notebook, click **Existing Jupyter Server** and paste the URL.

In [8]:
import sys
from pathlib import Path
BAMBOO_SETUP = Path.cwd()
NOTEBOOKSDIR = BAMBOO_SETUP / 'src' / 'post_processing' / 'NN' / 'notebooks'
sys.path.append(str((BAMBOO_SETUP/'src').resolve()))
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
# pd.set_option('display.max_colwidth', None)  # Allow columns to be fully displayed

In [2]:
Z_OUTPUT_eos = Path('/eos/user/a/anunezde/Z_OUTPUT_eos')
WORKDIR = Z_OUTPUT_eos / '2022_even_0822' / 'LLR_and_vars_4o5'
sel_name = 'SL_res_2b_x'
total_inputs = 'input/8llrs1D.txt'

# Setup a data handler

In [4]:
from post_processing.NN.DataHandler import DataHandler
datahandler = DataHandler(workdir=WORKDIR.resolve(), tree_name=sel_name, total_inputs=total_inputs)
total_df = datahandler.load_data()
total_df = datahandler.fix_mismatch(total_df)
total_df

	Loading data ...


,event,genWeight,bjet0_pt_llr,bjets_dEta_llr,bjets_dPhi_llr,bjets_dR_llr,bjets_mbb_llr,mjj_llr,trijet_mInv_llr,trijet_pt_rat_llr,File,Process
186577,18,3.805950,-0.278,0.401,0.742,1.496,1.167,0.428,0.439,0.014,tbarWplus_dl,tW
13167,32,0.033119,-0.242,0.329,0.937,1.507,0.849,0.393,0.549,1.259,bbWW_sl,HH_bbWW
706510,36,81.103897,-0.322,-0.791,-1.330,-2.183,-2.512,-0.028,0.317,-0.649,TTbar_dl,ttbar
13168,42,0.033119,0.132,0.239,1.084,1.504,0.714,0.130,0.117,-0.098,bbWW_sl,HH_bbWW
11905,58,0.033119,0.109,0.358,0.747,1.493,1.051,-0.491,0.396,-0.674,bbWW_dl,HH_bbWW
...,...,...,...,...,...,...,...,...,...,...,...,...
30323,744899478,83745.546875,-0.288,0.327,-0.377,-0.090,-0.269,-0.151,-0.445,-0.622,Wjets_2J,WJets
30340,745108688,83745.546875,-0.223,0.343,0.757,1.520,0.263,0.193,0.558,-0.775,Wjets_2J,WJets
30318,745244862,83745.546875,-0.219,NaN,0.886,-1.550,-inf,-0.071,-1.004,0.563,Wjets_2J,WJets
30006,745282826,-83745.546875,-0.325,-0.230,0.668,-0.090,0.155,-0.419,-0.551,-0.246,Wjets_2J,WJets


In [5]:
preprocessed_df = datahandler.preprocess_data(total_df, nan_replacement = -9999)


Preprocessing data ...
After removing events with negative weights:
LLRs were found in the loaded data.
Replacing nan values with -9999
Total_df:
            event     genWeight  bjet0_pt_llr  bjets_dEta_llr  bjets_dPhi_llr  bjets_dR_llr  bjets_mbb_llr  mjj_llr  trijet_mInv_llr  trijet_pt_rat_llr          File  Process_DY  Process_HH_bbWW  Process_HH_bbtautau  Process_VV  Process_WJets  Process_tW  Process_ttbar
186577         18      3.805950        -0.278           0.401           0.742         1.496          1.167    0.428            0.439              0.014  tbarWplus_dl           0                0                    0           0              0           1              0
13167          32      0.033119        -0.242           0.329           0.937         1.507          0.849    0.393            0.549              1.259       bbWW_sl           0                1                    0           0              0           0              0
706510         36     81.103897        -0.3

# Set up your model config

In [5]:
from post_processing.NN.utils import ModelConfig
model_config = ModelConfig(
    name='multi_HH_ttbar_tW',
    type='multi',
    categorization={"HH": ["HH_bbWW"], "ttbar": ["ttbar"], "tW": ["tW"]},
    training_weight_sf={"HH_bbWW": 1.0, "ttbar": 8.0, "tW": 4.0},
    input_vars='All',
    architecture_in_yml=False,
    residual_network=True,
    hiddenlayers=[
        {"type": 'Dense', "units": 256, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 256, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4},
        {"type": 'Dense', "units": 256, "activation": 'relu', "act_regularizer": {'l2': 1e-4}, "dropout_rate": 0.4}
    ],
    outputlayers=[
        {"type": 'Dense', "units": 3, "kernel_initializer": 'normal', "activation": 'softmax', "act_regularizer": {'l2': 1e-4}, "name": 'output'}
    ],
    compiler={"optimizer": 'adam', "lr": 0.001, "loss": 'categorical_crossentropy'},
    fit={"batch_size": 1024, "epochs": 100, "validation_split": 0.25}
)

# DNNModel

In [ ]:
from post_processing.NN.DNNModel import DNNModel
DNN = DNNModel(model_config=model_config, modeldir= NOTEBOOKSDIR/'model1')
model_df = DNN.set_model_df_from_total_df(total_df)
X_train, X_test, Y_train, Y_test, evs_test, sw_train = DNN.Full_Splitting(model_df)
input_layer, normalized_input = DNNModel.input_preprocessing(X_train)

In [ ]:
Y_train_for_binary = Y_train['Class_ttbar']
Y_train_for_multi = Y_train
Y_test_for_binary = Y_test['Class_ttbar']
Y_test_for_multi = Y_test
Y_train = {'binary_output': Y_train_for_binary, 'multiclass_output': Y_train_for_multi}
Y_test = {'binary_output': Y_test_for_binary, 'multiclass_output': Y_test_for_multi}

### Train and Evaluate

In [44]:
def train_and_eval(DNN, tf_model=None):
    if tf_model is not None:
        DNN.model = tf_model
    else:
        DNN.build_model(input_layer=input_layer, normalized_input=normalized_input)
    DNN.train_model(X_train, Y_train, sw_train)
    DNN.Evaluate(X_test, Y_test, evs_test)

In [19]:
from tensorflow.keras import layers, models, Input, regularizers
import tensorflow as tf
from post_processing.NN.model_builder import get_metrics
from tensorflow.keras.layers import Layer

class LessThanThreshold(Layer):
    def __init__(self, threshold=0.5, **kwargs):
        super(LessThanThreshold, self).__init__(**kwargs)
        self.threshold = threshold

    def call(self, inputs):
        return tf.cast(tf.less(inputs, self.threshold), tf.float32)

    def compute_mask(self, inputs, mask=None):
        # Pass the existing mask through, or create a new one if needed
        return mask

def hierarchical_model(input_layer, normalized_input):
    reg_l2 = regularizers.l2(1e-4)
    x = normalized_input
    x = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(x)
    x = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(x)
    x = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(x)
    x = layers.Dropout(0.4)(x)

    binary_output = layers.Dense(1, activation='sigmoid', kernel_initializer = 'normal', activity_regularizer=reg_l2, name='binary_output')(x)
    
    # Use the custom layer in your model
    mask = LessThanThreshold(threshold=0.5)(binary_output)
    masked_input = layers.Multiply()([normalized_input, mask])

    y = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(masked_input)
    y = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(y)
    y = layers.Dense(32, activation='relu', activity_regularizer=reg_l2)(y)
    y = layers.Dropout(0.4) (x)

    multiclass_output = layers.Dense(3, activation='softmax', kernel_initializer = 'normal', activity_regularizer=reg_l2, name='multiclass_output')(y)

    model = models.Model(inputs = input_layer, outputs=[binary_output, multiclass_output], name='hierarchical_model')

    model.compile(
        optimizer='adam',
        loss={'binary_output': 'binary_crossentropy', 'multiclass_output': 'categorical_crossentropy'},
        metrics = {'binary_output': get_metrics(classes=['ttbar']), 'multiclass_output': get_metrics(classes=DNN.classes)}
        )

    return model

model = hierarchical_model(input_layer, normalized_input)
DNN.model = model

In [ ]:
DNN.train_model(X_train, Y_train, sw_train)

In [23]:
model_metrics = DNN.model.evaluate(X_test, Y_test, verbose=0, return_dict=True)   

In [ ]:
print(model_metrics)

In [ ]:
Y_pred_score = DNN.model.predict(X_test)

In [34]:
binary_predictions = Y_pred_score[0]
multi_predicitions = Y_pred_score[1]

In [ ]:
multi_predicitions.shape

In [ ]:
for y_test, y_pred in zip(Y_test.values(),Y_pred_score):
    print(y_test.shape)
    print(y_pred.shape)